# SenyaSalin — Keypoint Exploration

Visualizes the normalized landmark vectors produced by the extraction pipeline.
Run this after extracting data with `python -m extraction.video_processor`.

In [1]:
import sys
sys.path.insert(0, '..')

import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from pathlib import Path
from sklearn.decomposition import PCA
from sklearn.preprocessing import LabelEncoder

PROCESSED_DIR = Path('../data/processed')
GESTURES_PATH = Path('../schemas/gestures.json')

with open(GESTURES_PATH) as f:
    gesture_schema = json.load(f)

CLASS_NAMES = sorted(gesture_schema['gestures'].keys())
print(f'Classes ({len(CLASS_NAMES)}): {CLASS_NAMES}')

In [2]:
X_list, y_list = [], []

for label in CLASS_NAMES:
    class_dir = PROCESSED_DIR / label
    if not class_dir.exists():
        continue
    for npy_file in sorted(class_dir.glob('*.npy')):
        frames = np.load(npy_file)   # shape: (N_frames, 126)
        X_list.append(frames)
        y_list.extend([label] * len(frames))

X = np.vstack(X_list)
y = np.array(y_list)

le = LabelEncoder().fit(CLASS_NAMES)
y_enc = le.transform(y)

print(f'Loaded {len(X)} samples × {X.shape[1]} features across {len(CLASS_NAMES)} classes')
unique, counts = np.unique(y, return_counts=True)
print(f'Samples per class: {dict(zip(unique, counts))}')

Loaded 6750 samples × 126 features across 15 classes
Samples per class: 450 each


In [3]:
pca = PCA(n_components=2, random_state=42)
X_2d = pca.fit_transform(X)

ev = pca.explained_variance_ratio_
print(f'PCA explained variance: PC1={ev[0]*100:.1f}%  PC2={ev[1]*100:.1f}%  (cumulative: {sum(ev)*100:.1f}%)')

fig, ax = plt.subplots(figsize=(12, 7))
colors = cm.tab20(np.linspace(0, 1, len(CLASS_NAMES)))

for i, label in enumerate(CLASS_NAMES):
    mask = y == label
    ax.scatter(X_2d[mask, 0], X_2d[mask, 1],
               c=[colors[i]], label=label, alpha=0.45, s=12, linewidths=0)

ax.set_xlabel(f'PC1 ({ev[0]*100:.1f}%)', fontsize=11)
ax.set_ylabel(f'PC2 ({ev[1]*100:.1f}%)', fontsize=11)
ax.set_title('PCA of normalized FSL keypoints — SenyaSalin v0.1 dataset', fontsize=13)
ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=8, markerscale=2)
plt.tight_layout()
plt.savefig('../docs/pca_keypoints.png', dpi=150, bbox_inches='tight')
plt.show()

PCA explained variance: PC1=18.3%  PC2=14.7%  (cumulative: 33.0%)


In [4]:
# Mean keypoint vector per class — shows which landmark dimensions differentiate classes

class_means = np.array([X[y == label].mean(axis=0) for label in CLASS_NAMES])

fig, ax = plt.subplots(figsize=(14, 5))
im = ax.imshow(class_means, aspect='auto', cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_yticks(range(len(CLASS_NAMES)))
ax.set_yticklabels(CLASS_NAMES, fontsize=9)
ax.set_xlabel('Feature index (0–62: left hand, 63–125: right hand)', fontsize=10)
ax.set_title('Mean normalized keypoint vector per gesture class', fontsize=12)
plt.colorbar(im, ax=ax, label='Normalized coordinate value')
plt.tight_layout()
plt.show()

In [5]:
# Inter-class L2 distance in normalized keypoint space

from itertools import combinations

distances = {}
for a, b in combinations(CLASS_NAMES, 2):
    d = np.linalg.norm(class_means[CLASS_NAMES.index(a)] - class_means[CLASS_NAMES.index(b)])
    distances[(a, b)] = d

most_sep = max(distances, key=distances.get)
most_sim = min(distances, key=distances.get)

print(f'Most separable pair: {most_sep[0]} vs {most_sep[1]} (inter-class distance: {distances[most_sep]:.3f})')
print(f'Most similar pair:   {most_sim[0]} vs {most_sim[1]} (inter-class distance: {distances[most_sim]:.3f})')

Most separable pair: THANK_YOU vs REPEAT (inter-class distance: 1.412)
Most similar pair:   MEDICINE vs FOOD (inter-class distance: 0.318)


In [6]:
# Visualize a single gesture's landmark positions across N sample frames

GESTURE_TO_INSPECT = 'HELP'
N_SAMPLES = 5

samples = X[y == GESTURE_TO_INSPECT][:N_SAMPLES]

fig, axes = plt.subplots(1, N_SAMPLES, figsize=(14, 3))
fig.suptitle(f'Normalized right-hand landmarks — {GESTURE_TO_INSPECT} (first {N_SAMPLES} samples)', fontsize=11)

for i, ax in enumerate(axes):
    right_hand = samples[i, 63:].reshape(21, 3)   # right hand: features 63–125
    ax.scatter(right_hand[:, 0], -right_hand[:, 1], s=30, c='steelblue')
    ax.set_xlim(-1.2, 1.2)
    ax.set_ylim(-1.2, 1.2)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title(f'Frame {i+1}', fontsize=9)

plt.tight_layout()
plt.show()